# Analisis ulasan dan rekomendasi wisata Madura

**Malik Hariyanto · Asistensi Hibah Penelitian**

Pipeline untuk proposal *Smart Tourism Recommender System Based on Seasonal Context Aware Using FastText for Personalized Destinations*. Jalankan bagian 00 sampai 10 dari atas. Setiap bagian menjelaskan masukan, proses, dan keluaran. Implementasi fungsi dapat dipelajari di folder `src/wisata_madura/`.

| Dataset sekarang | Implikasi |
|---|---|
| Workbook memuat lokasi seperti Kebun Binatang Surabaya dan tidak memiliki kolom kabupaten | Isi CSV verifikasi wilayah sebelum menarik kesimpulan khusus Madura. |
| Label `rekomendasi_cuaca` dan `wisata` tersedia per ID destinasi | Dua atribut ini menjadi target klasifikasi. |
| Kolom `date` berisi tanggal relatif, dan anotasi sentimen belum tersedia | Analisis musim kunjungan serta klasifikasi sentimen memerlukan data tambahan. |
| Penilaian relevansi pengguna disiapkan terpisah | BGE dapat menghasilkan peringkat. Precision@K dan Recall@K dihitung setelah penilaian tersedia. |

## 00 Konfigurasi dan instalasi

Data siap-model sudah ikut dalam repo dan dimuat otomatis. Untuk menjalankan dataset bawaan, biarkan `DATA_PATH = None`. Isi path hanya jika ingin mencoba workbook lain. Aktifkan GPU untuk pelatihan BiLSTM. Setel `RUN_DEEP = False` jika ingin menjalankan jalur dasar lebih dulu. Setel `RUN_BGE = True` hanya saat siap mengunduh model BGE-M3.

In [ ]:
from datetime import datetime
from pathlib import Path

DATA_PATH = None  # Pakai dataset bawaan repo secara otomatis.
REGION_PATH = None  # Contoh: Path("wilayah_destinasi.csv")
OUTPUT_DIR = Path("hasil_wisata_madura") / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
JUDGMENTS_PATH = None  # Contoh: Path("relevansi_destinasi.csv")
SEED = 42
TEST_FRACTION = 0.20
N_TOPICS = 6
RUN_DEEP = True
RUN_BGE = False

In [ ]:
import subprocess
import sys

cwd = Path.cwd()
checkout = cwd / "wisata_madura_bundled_v1"
candidates = (cwd, cwd.parent, checkout)
repo_root = next((
    p for p in candidates
    if (p / "src" / "wisata_madura").is_dir() and (p / "data" / "manifest.json").is_file()
), None)

if repo_root is None:
    if checkout.exists():
        raise FileExistsError(f"Perbarui checkout {checkout} dari cabang riset, lalu jalankan sel lagi.")
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "research/audit-wisata-madura",
        "https://github.com/MalikHariyanto/crispy_recom_machine_system.git", str(checkout),
    ], check=True)
    repo_root = checkout

requirements = ["-r", str(repo_root / "requirements-wisata.txt")]
if RUN_DEEP:
    requirements += ["gensim>=4.4,<5", "torch>=2.2,<3"]
if RUN_BGE:
    requirements += ["sentence-transformers>=3,<6"]
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements], check=True)
sys.path.insert(0, str(repo_root / "src"))

## 01 Muat dataset otomatis dan periksa skema

**Masukan:** paket di folder `data/`, dibentuk dari lembar **Data Ulasan** pada `koreksi_dataset.xlsx`. Kode memeriksa checksum, jumlah baris, dan konsistensi label sebelum data masuk ke model. Untuk cakupan Madura, isi kabupaten pada templat CSV dan pasang `REGION_PATH`. Baris dengan kabupaten kosong dianggap belum terverifikasi. Tanpa CSV, keluaran mendeskripsikan workbook campuran.

Paket memuat teks model dari kolom `text_preprocessed`, nama dan ID destinasi, label, serta penanda audit. Teks asli, rating, dan nilai tanggal Excel tidak diperlukan untuk pelatihan. Normalisasi ringan mempertahankan negasi seperti *tidak* atau *kurang*.

In [ ]:
import json
from importlib.metadata import version

import pandas as pd
from IPython.display import display

from wisata_madura.data import (
    audit_bundled_reviews, audit_reviews, label_distribution, load_bundled_reviews,
    load_reviews, region_template,
    select_madura_reviews, split_reviews,
)
from wisata_madura.modeling import evaluate_classifiers, fit_topics, run_baselines, run_lda_only

if DATA_PATH is None:
    source_reviews, data_manifest = load_bundled_reviews(repo_root / "data")
    print("Dataset bawaan repo berhasil dimuat.")
else:
    data_manifest = None
    source_reviews = load_reviews(Path(DATA_PATH))
    print(f"Dataset alternatif dimuat dari {DATA_PATH}.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
region_template(source_reviews).to_csv(OUTPUT_DIR / "template_verifikasi_wilayah.csv", index=False)
if REGION_PATH is None:
    reviews = source_reviews
    geography = {
        "scope": "workbook_complete_unverified",
        "source_destinations": source_reviews["id_destinasi"].nunique(),
    }
    print("Cakupan: workbook campuran. Isi REGION_PATH untuk analisis khusus Madura.")
else:
    reviews, geography = select_madura_reviews(source_reviews, REGION_PATH)
    print("Cakupan: ID destinasi Madura yang telah diverifikasi pada CSV.")
print(f"{len(reviews):,} ulasan dari {reviews['id_destinasi'].nunique()} destinasi")
print(list(reviews.columns))

## 02 Audit kualitas dan distribusi label

**Keluaran:** jumlah teks kosong, duplikat persis, tanggal relatif, indikator karakter rusak, dan distribusi atribut. Indikator karakter rusak membantu memilih sampel untuk diperiksa. Pada paket bawaan, hitungan tanggal relatif dan karakter rusak menggunakan penanda yang dihitung dari workbook sumber.

In [ ]:
if data_manifest is None:
    audit = audit_reviews(reviews, Path(DATA_PATH))
else:
    audit = audit_bundled_reviews(reviews, data_manifest)
audit["geography"] = geography
distribution = label_distribution(reviews)
display(pd.Series({key: audit[key] for key in (
    "rows", "destinations", "exact_duplicate_rows", "empty_normalized_text_rows",
    "relative_date_rows", "potential_encoding_error_rows",
)}))
display(distribution)
distribution.to_csv(OUTPUT_DIR / "distribusi_label.csv", index=False)

## 03 Pisahkan data latih dan uji

**Keluaran:** partisi dengan ID destinasi terpisah. Sekitar 20% ID per jenis wisata dialokasikan ke uji. Jumlah ulasan dapat berbeda dari rasio 80:20 karena satu destinasi memiliki banyak ulasan. Ulasan kosong dan pengulangan dalam satu destinasi disisihkan. Teks uji yang juga muncul di latih dikeluarkan agar contoh identik tidak muncul di kedua sisi.

In [ ]:
split = split_reviews(reviews, test_fraction=TEST_FRACTION, seed=SEED)
assert set(split.train["id_destinasi"]).isdisjoint(split.test["id_destinasi"])
assert set(split.train["teks_model"]).isdisjoint(split.test["teks_model"])
audit.update(split.audit)
display(pd.Series(split.audit))

## 04 Latih model pembanding

**Keluaran:** prediksi model mayoritas dan TF-IDF + Linear SVC untuk dua atribut destinasi. Model mayoritas memberi nilai acuan. TF-IDF menghitung bobot kata dari korpus latih. Kedua model memakai partisi yang sama dengan skenario berikutnya.

In [ ]:
predictions = run_baselines(split, seed=SEED)
print("Model pembanding:", list(predictions))

## 05 Ekstrak topik LDA dan uji fitur topik

**Keluaran:** distribusi topik per ulasan dan daftar kata utama. Satu LDA dilatih pada sampel ulasan latih dengan maksimum 30 ulasan per destinasi. Model bersama menjaga arti tiap dimensi topik konsisten tanpa memilih model berdasarkan label yang hendak diprediksi.

Ringkasan topik per label cuaca bersifat deskriptif untuk kategori destinasi. Skenario LDA saja memakai regresi logistik karena vektor topik merupakan fitur tabular.

In [ ]:
topics = fit_topics(split, n_topics=N_TOPICS, seed=SEED)
predictions.update(run_lda_only(split, topics, seed=SEED))
audit.update({"lda_sample_reviews": topics.sample_size, "lda_topics": N_TOPICS})
display(topics.keywords)
display(topics.weather_summary.round(3))
topics.keywords.to_csv(OUTPUT_DIR / "topik_lda.csv", index=False)
topics.weather_summary.to_csv(OUTPUT_DIR / "ringkasan_topik_per_label_cuaca.csv")

## 06 Latih FastText pada korpus latih

**Keluaran:** matriks vektor kata dan urutan ID token. FastText mempelajari potongan karakter dari teks latih. Kata pada data uji yang belum pernah muncul tetap dapat memperoleh vektor dari potongan karakter. ID nol menandai padding. Ulasan lebih panjang dari `max_tokens` dipotong setelah token ke-64 dengan konfigurasi awal.

In [ ]:
if RUN_DEEP:
    from dataclasses import asdict
    from wisata_madura.neural import DeepConfig, fit_fasttext, run_bilstm_variants

    deep_config = DeepConfig(seed=SEED)
    encoded = fit_fasttext(split, deep_config)
    audit["deep_config"] = asdict(deep_config)
    print("Ukuran urutan latih:", encoded.train_tokens.shape)
    print("Ukuran matriks embedding:", encoded.embeddings.shape)
else:
    print("FastText dilewati. Aktifkan RUN_DEEP di bagian 00 untuk menjalankannya.")

## 07 Bandingkan dua varian BiLSTM

**Keluaran:** prediksi untuk skenario FastText saja dan FastText + LDA, serta catatan loss latih. BiLSTM membaca urutan kata dari dua arah, lalu satu kepala klasifikasi memprediksi setiap atribut. Pada varian gabungan, distribusi topik ditambahkan ke representasi urutan sebelum klasifikasi. Kedua varian menggunakan jumlah epoch yang sama. Data uji baru dipakai setelah pelatihan.

In [ ]:
if RUN_DEEP:
    deep_predictions, training_history = run_bilstm_variants(split, topics, encoded, deep_config)
    predictions.update(deep_predictions)
    training_history.to_csv(OUTPUT_DIR / "riwayat_bilstm.csv", index=False)
    display(training_history)
else:
    print("BiLSTM dilewati. Bagian evaluasi tetap menampilkan model yang sudah dijalankan.")

## 08 Evaluasi pada tingkat ulasan dan destinasi

**Keluaran:** accuracy, micro-F1, macro-F1, Hamming loss, dan tabel per kelas. Prediksi per destinasi menggabungkan prediksi ulasan melalui suara terbanyak. Macro-F1 memberi bobot setara pada kelas yang hadir di data uji. Selalu baca jumlah contoh per kelas sebelum menafsirkan skor.

In [ ]:
metrics, per_class = evaluate_classifiers(split.test, predictions)
display(metrics.loc[metrics["unit"].eq("destinasi")].round(4))
metrics.to_csv(OUTPUT_DIR / "metrik_klasifikasi.csv", index=False)
per_class.to_csv(OUTPUT_DIR / "metrik_per_kelas.csv", index=False)

## 09 Rekomendasikan destinasi dengan BGE

**Masukan:** preferensi, konteks Kemarau/Penghujan jika diketahui, dan ID kunjungan opsional. BGE-M3 memberi skor kemiripan semantik pada deskripsi destinasi. Atribut cuaca pada workbook menyaring kandidat sesuai pilihan pengguna. Modul memakai katalog yang tersedia sebagai tahap pemeringkatan. Keluaran klasifikasi dapat dipertimbangkan untuk katalog baru setelah modelnya tervalidasi.

Ubah preferensi dan musim di sel berikut. Peringkat contoh tidak digunakan sebagai penilaian relevansi. Isi `JUDGMENTS_PATH` dengan CSV relevansi independen berkolom `query_id,preferensi,musim,id_destinasi` untuk menghitung Precision@K dan Recall@K.

In [ ]:
from wisata_madura.recommendation import BGERecommender, build_catalog, evaluate_recommendations

PREFERENSI = "Wisata alam dengan pemandangan yang cocok untuk keluarga"
MUSIM = "Penghujan"
RIWAYAT_IDS = []
K = 5

catalog = build_catalog(reviews)
print(f"{len(catalog)} destinasi dalam katalog dengan cakupan {geography['scope']}.")

if RUN_BGE:
    recommender = BGERecommender.from_pretrained(catalog)
    ranking = recommender.recommend(PREFERENSI, MUSIM, k=K, riwayat_ids=RIWAYAT_IDS)
    display(ranking)
    ranking.assign(cakupan=geography["scope"]).to_csv(OUTPUT_DIR / "contoh_rekomendasi.csv", index=False)
    if JUDGMENTS_PATH is not None:
        judgments = pd.read_csv(JUDGMENTS_PATH, dtype={"id_destinasi": str})
        ranking_metrics = evaluate_recommendations(recommender, judgments, k=K)
        display(ranking_metrics)
        display(ranking_metrics[["precision_at_k", "recall_at_k"]].mean())
        ranking_metrics.to_csv(OUTPUT_DIR / "metrik_rekomendasi.csv", index=False)
    else:
        print("Precision@K dan Recall@K memerlukan CSV penilaian relevansi independen.")
else:
    print("Katalog siap. Aktifkan RUN_BGE di bagian 00 untuk memeringkat destinasi.")

## 10 Simpan catatan eksperimen

Folder keluaran berisi tabel agregat, hash sumber data, seed, dan versi perangkat lunak. Teks ulasan mentah tidak disalin ke keluaran. Templat wilayah hanya berisi ID, nama destinasi, dan isian kabupaten. Setiap eksekusi bagian 00 menyiapkan subfolder baru sehingga hasil percobaan dapat dipisahkan.

In [ ]:
packages = ["numpy", "pandas", "scikit-learn", "openpyxl"]
if RUN_DEEP:
    packages += ["gensim", "torch"]
if RUN_BGE:
    packages += ["sentence-transformers"]

audit.update({
    "python": sys.version.split()[0],
    "packages": {package: version(package) for package in packages},
    "run_deep": RUN_DEEP,
    "run_bge": RUN_BGE,
})
(OUTPUT_DIR / "audit_dataset.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Keluaran:", OUTPUT_DIR.resolve())
print("\n".join(f.name for f in sorted(OUTPUT_DIR.iterdir()) if f.is_file()))